In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS taxi_catalog.log_schema;

CREATE TABLE IF NOT EXISTS taxi_catalog.log_schema.log_error_details (
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    job_link STRING,
    stage STRING,
    status STRING,
    error_message STRING,
    run_id BIGINT,
    job_id BIGINT
)
USING DELTA;

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("operation", "log_start")
dbutils.widgets.text("stage_name", "JOB")
dbutils.widgets.text("error_message", "")

In [0]:
run_id = int(dbutils.widgets.get("run_id"))
job_id = int(dbutils.widgets.get("job_id"))

operation = dbutils.widgets.get("operation")
stage_name = dbutils.widgets.get("stage_name")
error_message = dbutils.widgets.get("error_message")

print("Run ID:", run_id)
print("Job ID:", job_id)
print("Operation:", operation)
print("Stage:", stage_name)
print("Error:", error_message)

In [0]:
workspace_url = "https://dbc-5c07a240-0780.cloud.databricks.com"

job_link = (
    f"{workspace_url}/jobs/{job_id}/runs/{run_id}"
    f"?o=7474644504186528"
)

print("Job Link:", job_link)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def write_log(stage, status, error=None):

    log_row = spark.range(1).select(
        current_timestamp().alias("start_date"),
        lit(None).cast("timestamp").alias("end_date"),
        lit(job_link).alias("job_link"),
        lit(stage).alias("stage"),
        lit(status).alias("status"),
        lit(error).alias("error_message"),
        lit(run_id).cast("long").alias("run_id"),
        lit(job_id).cast("long").alias("job_id")
    )

    if status in ["PASS", "FAIL"]:
        log_row = log_row.withColumn(
            "end_date", current_timestamp()
        )

    log_row.write.format("delta").mode("append").saveAsTable(
        "taxi_catalog.log_schema.log_error_details"
    )

    print(f"{stage} : {status} logged successfully")

In [0]:
if operation == "log_start":

    write_log(stage_name, "Started")

elif operation == "log_pass":

    write_log(stage_name, "PASS")

elif operation == "log_fail":

    write_log(stage_name, "FAIL", error_message)

else:
    raise ValueError(f"Invalid operation: {operation}")

In [0]:
%sql
SELECT *
FROM taxi_catalog.log_schema.log_error_details where start_date >= '2026-11-25T19:35:32.977+00:00'
ORDER BY start_date;